# Babaorum lecteur — R6 banc d'architectures, phase B4

Pré-enregistrement : docs/READER_R6_PREREG.txt (p9). Aucune ligne de test. La fumée ne mesure aucun CER.

In [ ]:
import os, sys, json, gzip, glob, hashlib, shutil, subprocess, time, zipfile, math, random, re, unicodedata
D, TMP, OUT = '/kaggle/input', '/kaggle/tmp', '/kaggle/working/r6'
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
PHASE = 'B4'          # 'SMOKE' ou un bras
STEPS = 14844            # S (fixé après la fumée, identique pour tous les bras)
SMOKE_STEPS = 300
EVAL_EVERY = 1000          # pas entre deux mesures sur TUNE (choix du point)
BATCH, LR, WD, WARMUP, CLIP, SEED = 32, 1e-3, 0.01, 1000, 1.0, 0
T_START = time.time()
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
%%bash
pip install -q rapidfuzz 2>&1 | tail -1

## Recadrages (comme E8 : archives publiques, empreinte MD5 contrôlée)

In [ ]:
def find(pattern):
    return sorted(glob.glob(f'{D}/**/{pattern}', recursive=True))
items_path = (find('e8_items.jsonl.gz') or find('e8_items.jsonl'))[0]
op = gzip.open if items_path.endswith('.gz') else open
with op(items_path, 'rt', encoding='utf-8') as fh:
    items = [json.loads(l) for l in fh if l.strip()]
sys.path.insert(0, os.path.dirname(find('crops.py')[0]))
from crops import crop_line
need = {}
for it in items:
    kind, name = it['image'].split('/', 1)
    need.setdefault(kind, set()).add(name)
print(len(items), 'lignes', {k: len(v) for k, v in need.items()})

NEWSEYE = 'https://zenodo.org/records/4293602/files/ATR_TrainingSet_BnF_Newseye_M2+.zip?download=1'
BNL = 'https://data.bnl.lu/wp-content/uploads/2021/07/bnl-ground-truth-newspapers-before-1878.zip'
def fetch(url, path):
    if not os.path.exists(path):
        subprocess.run(['wget', '-q', '-O', path, url], check=True)
def extract(zf, wanted, dest):
    os.makedirs(dest, exist_ok=True)
    for info in zf.infolist():
        base = os.path.basename(info.filename)
        if base in wanted and not os.path.exists(f'{dest}/{base}'):
            with zf.open(info) as s, open(f'{dest}/{base}', 'wb') as d:
                shutil.copyfileobj(s, d)
IMG = f'{TMP}/img'
if not all(os.path.exists(f'{IMG}/newseye/{n}') for n in need['newseye']):
    fetch(NEWSEYE, f'{TMP}/newseye.zip')
    with zipfile.ZipFile(f'{TMP}/newseye.zip') as zf:
        extract(zf, need['newseye'], f'{IMG}/newseye')
    os.remove(f'{TMP}/newseye.zip')
if not all(os.path.exists(f'{IMG}/bnl/{n}') for n in need['bnl']):
    fetch(BNL, f'{TMP}/bnl.zip')
    with zipfile.ZipFile(f'{TMP}/bnl.zip') as zf:
        inner = next(i for i in zf.infolist() if i.filename.endswith('/antiqua.zip'))
        with zf.open(inner) as s, open(f'{TMP}/antiqua.zip', 'wb') as d:
            shutil.copyfileobj(s, d)
    os.remove(f'{TMP}/bnl.zip')
    with zipfile.ZipFile(f'{TMP}/antiqua.zip') as zf:
        extract(zf, need['bnl'], f'{IMG}/bnl')
    os.remove(f'{TMP}/antiqua.zip')

In [ ]:
# Recadrages gardés en mémoire (niveaux de gris, ~2 Go) ; empreinte MD5 comparée à celle du Mac.
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
by_image = {}
for it in items:
    by_image.setdefault(it['image'], []).append(it)
CROPS, mism = {}, []
t0 = time.time()
for img, group in by_image.items():
    with Image.open(f'{IMG}/{img}') as raw_img:
        page = raw_img.convert('L')
    for it in group:
        im = page if img.startswith('bnl/') else crop_line(page, tuple(it['box']), it['polygon'], pad=4).image
        if hashlib.md5(im.tobytes()).hexdigest() != it['crop_md5']:
            mism.append(it['stem'])
        CROPS[it['stem']] = im.copy()
TRAIN = [it for it in items if it['split'] == 'train']
DEV = [it for it in items if it['split'] == 'dev']
print(len(TRAIN), len(DEV), 'md5 différents :', len(mism), round(time.time() - t0), 's')

In [ ]:
# --- axel/reader/metric.py (D-150), recopié par le générateur ---
"""Une seule mesure de CER pour les lecteurs (C3, D-422).

D-150 d'abord : NFC, puis normalisation du scorer HIPE (minuscules,
ligatures pliées, non-mot → espace, blancs compactés) avec ``ſ`` → ``s`` en
premier. Le brut (NFC, blancs compactés) est l'annexe. CER micro : somme
des distances d'édition sur somme des longueurs de vérité normalisées.

Ligne dont la vérité est **vide après normalisation** (``*``, ``. —``) :
elle ne compte ni dans les erreurs ni dans le support du CER (convention de
``scripts/e8_report.py``, celle des chiffres E8 et de D-280). Elle n'est pas
oubliée pour autant : :func:`score` rend à part le nombre de ces lignes et
le nombre de caractères que le lecteur y a écrits (une sortie sur une ligne
sans texte est une invention possible).
"""


import re
import unicodedata
from collections.abc import Callable, Iterable

from rapidfuzz.distance import Levenshtein

#: Pliages du scorer HIPE (D-11) ; ``ſ`` est replié avant, par :func:`d150`.
HIPE_TABLE = {
    "_": " ",
    "aͤ": "ä",
    "oͤ": "ö",
    "uͤ": "ü",
    "ß": "ss",
    "ꝛ": "r",
    "œ": "oe",
    "æ": "ae",
}


def normalize_hipe(text: str) -> str:
    """La normalisation du scorer HIPE : minuscules, pliages ß/ꝛ/œ/æ,
    tout non-mot devient une espace, blancs compactés. ``ſ`` est gardé."""
    text = text.lower()
    for src, dst in HIPE_TABLE.items():
        text = text.replace(src, dst)
    text = re.sub(r"[^\w]", " ", text)
    return " ".join(text.split())


def d150(text: str) -> str:
    """Texte normalisé selon D-150 : NFC, puis HIPE avec ``ſ`` replié (sans
    NFC, HIPE change un accent décomposé en espace : D-422)."""
    return normalize_hipe(unicodedata.normalize("NFC", text).replace("ſ", "s"))


def raw(text: str) -> str:
    """Texte brut de l'annexe : NFC, blancs compactés."""
    return " ".join(unicodedata.normalize("NFC", text).split())


NORMS: dict[str, Callable[[str], str]] = {"d150": d150, "raw": raw}


def lev(a: str, b: str) -> int:
    return int(Levenshtein.distance(a, b))


def line_errors(hyp: str | None, truth: str, mode: str = "d150") -> tuple[int, int]:
    """(erreurs, support) d'une ligne ; ``(0, 0)`` si la vérité normalisée
    est vide (la ligne sort du CER, voir :func:`score`)."""
    f = NORMS[mode]
    h, t = f(hyp or ""), f(truth)
    if not t:
        return 0, 0
    return lev(h, t), len(t)


def cer(pairs: Iterable[tuple[str | None, str]], mode: str = "d150") -> float:
    """CER micro en %, ``pairs`` = [(lecture, vérité)]."""
    e = n = 0
    for h, t in pairs:
        a, b = line_errors(h, t, mode)
        e += a
        n += b
    return 100 * e / n if n else 0.0


def score(pairs: Iterable[tuple[str | None, str]]) -> dict[str, float | int]:
    """D-150 d'abord, brut en annexe, et les lignes à vérité vide à part."""
    ps = list(pairs)
    out: dict[str, float | int] = {
        "cer_d150": cer(ps, "d150"),
        "cer_raw": cer(ps, "raw"),
        "lines": len(ps),
    }
    for mode in ("d150", "raw"):
        f = NORMS[mode]
        empty = [(h, t) for h, t in ps if not f(t)]
        out[f"empty_truth_lines_{mode}"] = len(empty)
        out[f"empty_truth_output_chars_{mode}"] = sum(len(f(h or "")) for h, _ in empty)
    return out


In [ ]:
import torch
# --- babaorum/reader/r6_arch.py, recopié par le générateur ---
"""R6 — architectures du banc de lecteurs (docs/READER_R6_PREREG.txt).

Chaque bras prend une ligne en niveaux de gris [B, 1, h_in, W] (blanc = 1,
remplissage de lot en blanc) ; l'image est inversée à l'entrée (fond = 0) pour
que le remplissage implicite des convolutions (zéros) soit du blanc : une
ligne lue seule ou en lot donne la même sortie. Le modèle
rend des log-probabilités CTC [B, T, C] avec T = W // pas. Décodeur CTC partout.
``ARMS[nom].describe()`` donne la liste exacte des couches (écrite dans le
pré-enregistrement avant la fumée). Aucune dépendance à axel.
"""


import math

import torch
from torch import nn


def conv_bn(cin: int, cout: int, k=3, s=1, p=1) -> nn.Sequential:
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, p, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class PosEnc1D(nn.Module):
    def __init__(self, d: int, max_len: int = 4000) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[: x.shape[1]][None]


def key_padding(lengths: torch.Tensor, T: int) -> torch.Tensor:
    return torch.arange(T, device=lengths.device)[None] >= lengths[:, None]


class BiLSTMHead(nn.Module):
    def __init__(self, d: int, hid: int, layers: int, n_out: int) -> None:
        super().__init__()
        self.rnn = nn.LSTM(d, hid, layers, bidirectional=True, batch_first=True, dropout=0.1)
        self.out = nn.Linear(2 * hid, n_out)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        y = nn.utils.rnn.pad_packed_sequence(self.rnn(packed)[0], batch_first=True, total_length=x.shape[1])[0]
        return self.out(y)


class ConformerBlock(nn.Module):
    """Demi-FFN, attention, convolution profondeur 15, demi-FFN (Gulati et al.)."""

    def __init__(self, d: int, heads: int = 4, k: int = 15) -> None:
        super().__init__()
        self.ff1 = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 4 * d), nn.SiLU(), nn.Dropout(0.1), nn.Linear(4 * d, d))
        self.ln_att = nn.LayerNorm(d)
        self.att = nn.MultiheadAttention(d, heads, dropout=0.1, batch_first=True)
        self.ln_conv = nn.LayerNorm(d)
        self.pw1 = nn.Conv1d(d, 2 * d, 1)
        self.dw = nn.Conv1d(d, d, k, padding=k // 2, groups=d)
        self.bn = nn.BatchNorm1d(d)
        self.pw2 = nn.Conv1d(d, d, 1)
        self.ff2 = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 4 * d), nn.SiLU(), nn.Dropout(0.1), nn.Linear(4 * d, d))
        self.ln_out = nn.LayerNorm(d)

    def forward(self, x: torch.Tensor, pad: torch.Tensor) -> torch.Tensor:
        x = x + 0.5 * self.ff1(x)
        h = self.ln_att(x)
        x = x + self.att(h, h, h, key_padding_mask=pad, need_weights=False)[0]
        h = self.ln_conv(x).masked_fill(pad[..., None], 0).transpose(1, 2)
        h = nn.functional.glu(self.pw1(h), dim=1)
        h = self.pw2(nn.functional.silu(self.bn(self.dw(h))))
        x = x + h.transpose(1, 2)
        x = x + 0.5 * self.ff2(x)
        return self.ln_out(x)


class Reader(nn.Module):
    """Squelette commun : tronc 2D -> repli de la hauteur -> [mélangeur] -> tête."""

    name = "?"
    h_in = 64
    stride = 4

    def describe(self) -> str:
        n = sum(p.numel() for p in self.parameters())
        return f"{self.name} h_in={self.h_in} pas={self.stride} trames/hauteur={self.h_in // self.stride} params={n}\n{self}"

    def lengths(self, widths: torch.Tensor) -> torch.Tensor:
        return torch.div(widths, self.stride, rounding_mode="floor").clamp(min=1)


class B0(Reader):
    """CRNN de type kraken : 4 conv 3×3 (32, 32, 64, 64) avec 2 max-pool 2×2 (pas 4)
    et 2 max-pool 2×1 (hauteur), repli de la hauteur par moyenne, BiLSTM 3×200."""

    name, h_in, stride = "B0", 64, 4

    def __init__(self, n_out: int) -> None:
        super().__init__()
        self.cnn = nn.Sequential(
            conv_bn(1, 32), nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
            conv_bn(32, 32), nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
            conv_bn(32, 64), nn.MaxPool2d((2, 1), (2, 1)), nn.Dropout2d(0.1),
            conv_bn(64, 64), nn.MaxPool2d((2, 1), (2, 1)), nn.Dropout2d(0.1),
        )
        self.proj = nn.Linear(64 * 4, 256)
        self.head = BiLSTMHead(256, 200, 3, n_out)

    def forward(self, x: torch.Tensor, widths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        f = self.cnn(1.0 - x)  # [B, 64, 4, W/4] ; inversion : fond = 0
        f = f.flatten(1, 2).transpose(1, 2)  # [B, W/4, 256]
        L = self.lengths(widths)
        return self.head(self.proj(f), L).log_softmax(-1), L


class ResTrunk(nn.Module):
    """ResNet-18 tronqué : conv 7×7 pas 2 + max-pool pas 2 (pas 4 en largeur),
    layer1 (64), layer2 (128, pas 2×1), layer3 (256, pas 2×1), sans layer4 ;
    hauteur 64 -> 4, repliée par moyenne. Sortie [B, W/4, 256]."""

    def __init__(self) -> None:
        super().__init__()

        def block(cin: int, cout: int, s) -> nn.Module:
            return _BasicBlock(cin, cout, s)

        self.stem = nn.Sequential(nn.Conv2d(1, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
                                  nn.MaxPool2d(3, 2, 1))
        self.layer1 = nn.Sequential(block(64, 64, 1), block(64, 64, 1))
        self.layer2 = nn.Sequential(block(64, 128, (2, 1)), block(128, 128, 1))
        self.layer3 = nn.Sequential(block(128, 256, (2, 1)), block(256, 256, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.layer3(self.layer2(self.layer1(self.stem(1.0 - x))))  # inversion : fond = 0
        return f.mean(2).transpose(1, 2)


class _BasicBlock(nn.Module):
    def __init__(self, cin: int, cout: int, s) -> None:
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, s, 1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)
        self.down = None if (cin == cout and s == 1) else nn.Sequential(nn.Conv2d(cin, cout, 1, s, bias=False),
                                                                        nn.BatchNorm2d(cout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = torch.relu(self.b1(self.c1(x)))
        y = self.b2(self.c2(y))
        return torch.relu(y + (x if self.down is None else self.down(x)))


class B1(Reader):
    """ResTrunk + BiLSTM 2×256."""

    name, h_in, stride = "B1", 64, 4

    def __init__(self, n_out: int) -> None:
        super().__init__()
        self.trunk = ResTrunk()
        self.head = BiLSTMHead(256, 256, 2, n_out)

    def forward(self, x, widths):
        L = self.lengths(widths)
        return self.head(self.trunk(x), L).log_softmax(-1), L


class B2(Reader):
    """ResTrunk + 2 couches transformer (d 256, 4 têtes, FFN 1 024, pré-norme,
    positions sinusoïdales, masque de remplissage) + BiLSTM 2×256."""

    name, h_in, stride = "B2", 64, 4

    def __init__(self, n_out: int) -> None:
        super().__init__()
        self.trunk = ResTrunk()
        self.pos = PosEnc1D(256)
        layer = nn.TransformerEncoderLayer(256, 4, 1024, 0.1, batch_first=True, norm_first=True)
        self.mix = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = BiLSTMHead(256, 256, 2, n_out)

    def mixed(self, x, widths):
        L = self.lengths(widths)
        f = self.trunk(x)
        return self.mix(self.pos(f), src_key_padding_mask=key_padding(L, f.shape[1])), L

    def forward(self, x, widths):
        f, L = self.mixed(x, widths)
        return self.head(f, L).log_softmax(-1), L


class B3(Reader):
    """SVTR léger : tronc conv 3×3 (32 pas 2, 64 pas 2, 128 pas 2) -> grille
    h 12 × W/8, 2 blocs MHSA globaux (d 128, 4 têtes) sur les jetons 2D avec
    positions sinusoïdales séparables, repli de la hauteur par moyenne, tête
    linéaire CTC (pas de BiLSTM)."""

    name, h_in, stride = "B3", 96, 8

    def __init__(self, n_out: int) -> None:
        super().__init__()
        self.stem = nn.Sequential(conv_bn(1, 32, s=2), conv_bn(32, 64, s=2), conv_bn(64, 128, s=2))
        layer = nn.TransformerEncoderLayer(128, 4, 512, 0.1, batch_first=True, norm_first=True)
        self.mix = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.pos_w = PosEnc1D(128)
        self.pos_h = nn.Parameter(torch.zeros(12, 128))
        self.out = nn.Sequential(nn.LayerNorm(128), nn.Linear(128, n_out))

    def forward(self, x, widths):
        f = self.stem(1.0 - x)  # [B, 128, 12, W/8] ; inversion : fond = 0
        B, C, H, W = f.shape
        L = self.lengths(widths)
        t = f.permute(0, 2, 3, 1)  # [B, H, W, C]
        t = t + self.pos_w.pe[:W][None, None] + self.pos_h[:H][None, :, None]
        pad = key_padding(L, W)[:, None].expand(B, H, W).reshape(B, H * W)
        t = self.mix(t.reshape(B, H * W, C), src_key_padding_mask=pad).reshape(B, H, W, C)
        return self.out(t.mean(1)).log_softmax(-1), L


class B4(B2):
    """B2 avec 2 blocs Conformer (d 256, 4 têtes, convolution 15) à la place du
    BiLSTM ; tête linéaire CTC."""

    name = "B4"

    def __init__(self, n_out: int) -> None:
        super().__init__(n_out)
        self.head = None
        self.conf = nn.ModuleList([ConformerBlock(256), ConformerBlock(256)])
        self.out = nn.Linear(256, n_out)

    def forward(self, x, widths):
        f, L = self.mixed(x, widths)
        pad = key_padding(L, f.shape[1])
        for blk in self.conf:
            f = blk(f, pad)
        return self.out(f).log_softmax(-1), L


class B5(B2):
    """B2 + tête CTC intermédiaire (linéaire) en sortie des couches transformer ;
    perte = 0,7·CTC final + 0,3·CTC intermédiaire ; sans réinjection. À
    l'inférence seule la sortie finale sert (score image seul)."""

    name = "B5"
    aux_weight = 0.3

    def __init__(self, n_out: int) -> None:
        super().__init__(n_out)
        self.aux = nn.Linear(256, n_out)

    def forward(self, x, widths, with_aux: bool = False):
        f, L = self.mixed(x, widths)
        main = self.head(f, L).log_softmax(-1)
        if with_aux:
            return main, L, self.aux(f).log_softmax(-1)
        return main, L


ARMS = {"B0": B0, "B1": B1, "B2": B2, "B3": B3, "B4": B4, "B5": B5}


In [ ]:
import numpy as np, torch, torch.nn as nn
from PIL import ImageFilter
DEV_T = 'cuda'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# codec : celui d'A1 (timing.json de la sortie axel-e17-vitctc) s'il est attaché, sinon caractères du train
_t = [p for p in sorted(glob.glob(f'{D}/**/timing.json', recursive=True)) if 'chars' in json.load(open(p))]
clean = lambda t: unicodedata.normalize('NFC', ' '.join(t.split()))
assert len(_t) <= 1, f'plusieurs timing.json avec chars : {_t}'
CHARS = json.load(open(_t[0]))['chars'] if _t else sorted({c for it in TRAIN for c in clean(it['truth'])})
print('codec :', _t[0] if _t else 'caractères du train', len(CHARS))
IDX = {c: i + 1 for i, c in enumerate(CHARS)}

# partition tune/eval de press/dev (graine 1, sha256 eval 996ab0cd…, READER_R3_PREREG)
def page_of(stem):
    return stem.split('__')[1] if '__' in stem else stem
pages = sorted({page_of(it['stem']) for it in DEV}); random.Random(1).shuffle(pages)
TUNE_P = set(pages[: len(pages) // 2])
TUNE = [it for it in DEV if page_of(it['stem']) in TUNE_P]
EVAL = [it for it in DEV if page_of(it['stem']) not in TUNE_P]
_h = lambda xs: hashlib.sha256('\n'.join(sorted(xs)).encode()).hexdigest()
print('tune', len(TUNE), 'eval', len(EVAL), 'sha eval pages', _h({page_of(it['stem']) for it in EVAL})[:8])

def augment(im, rng):
    # chacune avec probabilité 0,3 (pré-enregistrement)
    if rng.random() < 0.3:
        im = im.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.0)))
    if rng.random() < 0.3:
        a = np.asarray(im, np.float32) + np.random.default_rng(rng.randrange(1 << 30)).normal(0, rng.uniform(3, 12), (im.height, im.width))
        im = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    if rng.random() < 0.3:
        im = im.filter(ImageFilter.MinFilter(3) if rng.random() < 0.5 else ImageFilter.MaxFilter(3))
    if rng.random() < 0.3:
        im = im.rotate(rng.uniform(-2, 2), resample=Image.BILINEAR, expand=False, fillcolor=255)
    if rng.random() < 0.3:
        f = rng.uniform(-0.1, 0.1); d = int(round(abs(f) * im.height))
        if d > 0:
            if f > 0:
                c = Image.new('L', (im.width, im.height + 2 * d), 255); c.paste(im, (0, d)); im = c
            else:
                im = im.crop((0, d // 2, im.width, im.height - d // 2)) if im.height - d > 4 else im
    return im

def prep(im, h_in, stride):
    w = max(stride, round(im.width * h_in / max(1, im.height)))
    w = min(w, 4000)
    w = int(math.ceil(w / stride) * stride)
    g = im.convert('L').resize((w, h_in), Image.BICUBIC)
    return torch.from_numpy(np.asarray(g, np.float32) / 255.0)[None]

def batchify(ims):
    W = max(x.shape[-1] for x in ims)
    xb = torch.ones(len(ims), 1, ims[0].shape[1], W)
    for i, x in enumerate(ims):
        xb[i, :, :, : x.shape[-1]] = x
    return xb, torch.tensor([x.shape[-1] for x in ims])

def targets(its):
    ys = [[IDX[c] for c in clean(it['truth']) if c in IDX] for it in its]
    return ys

def greedy(lp):
    a = lp.argmax(-1).tolist(); out, prev = [], -1
    for c in a:
        if c != prev and c != 0:
            out.append(CHARS[c - 1])
        prev = c
    return ''.join(out)

@torch.no_grad()
def read(model, its, keep_logp=False):
    # évaluation LIGNE PAR LIGNE (règle R6 : indépendance au lot exacte)
    model.eval(); rows = []
    for it in its:
        x, w = batchify([prep(CROPS[it['stem']], model.h_in, model.stride)])
        with torch.autocast('cuda', dtype=torch.float16):
            lp, L = model(x.to(DEV_T), w.to(DEV_T))[:2]
        lp = lp[0, : int(L[0])].float().cpu()
        r = {'stem': it['stem'], 'truth': it['truth'], 'hyp': greedy(lp)}
        if keep_logp:
            r['logp'] = lp.half().numpy().tolist()
        rows.append(r)
    model.train()
    return rows

def cer_rows(rows):
    return score([(r['hyp'], r['truth']) for r in rows])['cer_d150']

def run(arm, steps, smoke=False):
    torch.manual_seed(SEED); rng = random.Random(SEED)
    model = ARMS[arm](len(CHARS) + 1).to(DEV_T)
    desc = model.describe()
    print(desc.split('\n')[0])
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / WARMUP if s < WARMUP else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - WARMUP) / max(1, steps - WARMUP)))))  # cosinus après l'échauffement
    scaler = torch.amp.GradScaler()
    ctc = nn.CTCLoss(zero_infinity=True)
    order = sorted(TRAIN, key=lambda it: CROPS[it['stem']].width / max(1, CROPS[it['stem']].height))
    chunks = [order[i:i + BATCH] for i in range(0, len(order), BATCH)]
    log = open(f'{OUT}/{arm}{"_smoke" if smoke else ""}_log.jsonl', 'w')
    best, step, t0, losses, inf_lines = 1e9, 0, time.time(), [], 0
    while step < steps:
        rng.shuffle(chunks)
        for b in chunks:
            if step >= steps:
                break
            ys = targets(b)
            keep = [i for i, y in enumerate(ys) if y]
            b = [b[i] for i in keep]; ys = [ys[i] for i in keep]
            x, w = batchify([prep(augment(CROPS[it['stem']], rng), model.h_in, model.stride) for it in b])
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(x.to(DEV_T), w.to(DEV_T), with_aux=True) if arm == 'B5' else model(x.to(DEV_T), w.to(DEV_T))
            lp, L = out[0].float(), out[1]
            tg = torch.tensor([c for y in ys for c in y]); tl = torch.tensor([len(y) for y in ys])
            inf_lines += int((L.cpu() < tl).sum())
            loss = ctc(lp.transpose(0, 1), tg, L.cpu(), tl)
            if arm == 'B5':
                loss = (1 - model.aux_weight) * loss + model.aux_weight * ctc(out[2].float().transpose(0, 1), tg, L.cpu(), tl)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP); scaler.step(opt); scaler.update(); sched.step()
            step += 1; losses.append(loss.item())
            if step % 100 == 0:
                row = {'step': step, 'loss': float(np.mean(losses[-100:])), 'steps_per_s': step / (time.time() - t0),
                       'lines_T_lt_U': inf_lines}
                log.write(json.dumps(row) + '\n'); log.flush(); print(row, flush=True)
            if not smoke and step % EVAL_EVERY == 0:
                c = cer_rows(read(model, TUNE))
                log.write(json.dumps({'step': step, 'tune_cer_d150': c}) + '\n'); log.flush(); print('tune', step, c, flush=True)
                if c < best - 1e-9:
                    best = c; torch.save(model.state_dict(), f'{OUT}/{arm}_best.pt')
    res = {'arm': arm, 'steps': step, 'seconds': time.time() - t0, 'steps_per_s': step / (time.time() - t0),
           'params': sum(p.numel() for p in model.parameters()), 'describe': desc, 'lines_T_lt_U': inf_lines}
    if not smoke:
        model.load_state_dict(torch.load(f'{OUT}/{arm}_best.pt'))
        tune_rows = read(model, TUNE, keep_logp=True); eval_rows = read(model, EVAL, keep_logp=True)
        res.update({'best_tune_cer_d150': best, 'tune_cer_d150': cer_rows(tune_rows), 'eval_cer_d150': cer_rows(eval_rows)})
        with gzip.open(f'{OUT}/{arm}_dev.jsonl.gz', 'wt') as g:
            for r in tune_rows + eval_rows:
                g.write(json.dumps(r, ensure_ascii=False) + '\n')
    json.dump(res, open(f'{OUT}/{arm}{"_smoke" if smoke else ""}_result.json', 'w'), indent=1)
    return res

if PHASE == 'SMOKE':
    out = [run(a, SMOKE_STEPS, smoke=True) for a in ('B0', 'B3')]
    print(json.dumps([{k: v for k, v in r.items() if k != 'describe'} for r in out], indent=1))
else:
    r = run(PHASE, STEPS)
    print(json.dumps({k: v for k, v in r.items() if k != 'describe'}, indent=1))